In [2]:
import rospy
import rosbag
import sensor_msgs.point_cloud2 as pc2
from sensor_msgs.msg import PointCloud2, PointField
from std_msgs.msg import Header
import numpy as np
import pandas as pd
import os
from datetime import datetime
import re
from pathlib import Path

In [50]:
nanosec_pattern = re.compile("(\d{4}-\d{1,2}-\d{1,2} \d{1,2}:\d{1,2}:\d{1,2}\.\d{1,6})(\d*)")

In [144]:
data_dir = Path('/root/data')
points_dir = data_dir/'velodyne_points/data/'
points_files = [file for file in sorted(points_dir.iterdir())]
output_rosbag_path = data_dir/'output.bag'
topic_name = 'velodyne_points'

In [88]:
def date2rostime(time: str):
    sec_match = nanosec_pattern.match(time)
    if sec_match is None:
        raise ValueError
    micro_part = sec_match.group(1)
    nano_part = sec_match.group(2)
    micro_date = datetime.strptime(micro_part, "%Y-%m-%d %H:%M:%S.%f")
    secs = int(micro_date.timestamp())
    nano_secs = micro_date.microsecond * 1000
    if nano_part is not None and nano_part != '':
        nano_secs += int(nano_part)
    return rospy.Time(secs, nano_secs)

In [ ]:
def read_points_from_file(path):
    points = pd.read_csv(path, sep=' ', header=None, dtype="float32")
    return points.apply(tuple, axis=1).tolist()

def read_timestamps(path):
    timestamps = pd.read_csv(Path(path)/'velodyne_points'/'timestamps.txt', header=None)
    timestamps_start = pd.read_csv(Path(path)/'velodyne_points'/'timestamps_start.txt', header=None)
    timestamps_end = pd.read_csv(Path(path)/'velodyne_points'/'timestamps_end.txt', header=None)
    all_timestamps = pd.concat([timestamps_start, timestamps, timestamps_end], axis=1)
    all_timestamps.columns = ["start", "mid", "end"]
    return all_timestamps.apply(lambda row: row.apply(date2rostime))

def interpolation_timestamps(idx: int, start: rospy.Time, end: rospy.Time, points_num: int):
    return start + (end - start) * idx / (points_num - 1)

def create_pointscloud_xyzi(points, timestamp, frame_id):
    header = Header(stamp=timestamp, frame_id=frame_id)
    dtype = PointField.FLOAT32
    fields = [PointField(name='x', offset=0, datatype=dtype, count=1),
                PointField(name='y', offset=4, datatype=dtype, count=1),
                PointField(name='z', offset=8, datatype=dtype, count=1),
                PointField(name='intensity', offset=12, datatype=dtype, count=1),]
    return pc2.create_cloud(header, fields, points)

def gen_rosbag(path):
    timestamps = read_timestamps(path)
    
    with rosbag.Bag(output_rosbag_path, 'w') as bag:
        for idx, row in timestamps.iterrows():
            points = read_points_from_file(points_files[idx])
            pointcloud = create_pointscloud_xyzi(points, row[1], 'velodyne')
            bag.write(topic_name, pointcloud)
            print(f'scan {idx + 1} was writed')

In [151]:
gen_rosbag(data_dir)

scan 1 was writed
scan 2 was writed
scan 3 was writed
scan 4 was writed
scan 5 was writed
scan 6 was writed
scan 7 was writed
scan 8 was writed
scan 9 was writed
scan 10 was writed
scan 11 was writed
scan 12 was writed
scan 13 was writed
scan 14 was writed
scan 15 was writed
scan 16 was writed
scan 17 was writed
scan 18 was writed
scan 19 was writed
scan 20 was writed
scan 21 was writed
scan 22 was writed
scan 23 was writed
scan 24 was writed
scan 25 was writed
scan 26 was writed
scan 27 was writed
scan 28 was writed
scan 29 was writed
scan 30 was writed
scan 31 was writed
scan 32 was writed
scan 33 was writed
scan 34 was writed
scan 35 was writed
scan 36 was writed
scan 37 was writed
scan 38 was writed
scan 39 was writed
scan 40 was writed
scan 41 was writed
scan 42 was writed
scan 43 was writed
scan 44 was writed
scan 45 was writed
scan 46 was writed
scan 47 was writed
scan 48 was writed
scan 49 was writed
scan 50 was writed
scan 51 was writed
scan 52 was writed
scan 53 was writed
sc